In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from service import SeattleBuildingService
from BuildingInput import BuildingInput


In [2]:
service = SeattleBuildingService()


In [3]:
df_esti = pd.read_csv('./data/projet6_estimation.csv')

In [4]:
#row_dic = df_esti.loc[0].to_dict()
#input_obj = BuildingInput (**row.to_dict() )

#predictions = [service.predict ( BuildingInput (**row.to_dict() ) ) ["prediction_tCO2"] for _, row in df_esti.iterrows()]
#predictions = service.predict (row_dic )
'''
predictions = [
    service.predict(BuildingInput(**row))["prediction_tCO2"]
    for row in df_esti.to_dict(orient="records")
]
'''
predictions = [
    service.predict(BuildingInput(**row))
    for row in df_esti.to_dict(orient="records")
]




In [5]:
# 2. Assignation des prédictions dans le DataFrame de test
data_test = df_esti.copy()
data_test["prediction"] = predictions


In [6]:
data_test["TotalGHGEmissions"].describe()

count     96.000000
mean      81.583750
std       81.843537
min        2.340000
25%       22.115000
50%       52.415000
75%       94.692500
max      359.090000
Name: TotalGHGEmissions, dtype: float64

In [7]:
data_test["SiteEnergyUse(kBtu)"].describe()

count    9.600000e+01
mean     2.708041e+06
std      2.600367e+06
min      0.000000e+00
25%      1.404300e+06
50%      1.982446e+06
75%      2.899628e+06
max      1.356777e+07
Name: SiteEnergyUse(kBtu), dtype: float64

In [8]:
data_test.describe()

,DataYear,Latitude,Longitude,NumberofBuildings,PropertyGFATotal,PropertyGFAParking,SiteEnergyUse(kBtu),TotalGHGEmissions,BuildingAge,mean_GFA_per_floor,Number_of_Use_Types
count,96.0,96.000000,96.000000,96.0,96.000000,96.000000,9.600000e+01,96.000000,96.000000,96.000000,96.000000
mean,2016.0,47.608172,-122.324150,1.0,80256.531250,181.989583,2.708041e+06,81.583750,45.656250,26867.420461,1.083333
std,0.0,0.064286,0.039907,0.0,60057.608983,1783.126471,2.600367e+06,81.843537,27.385677,17618.278683,0.374634
min,2016.0,47.499170,-122.407400,1.0,12294.000000,0.000000,0.000000e+00,2.340000,6.000000,4568.000000,1.000000
25%,2016.0,47.552343,-122.358735,1.0,43243.250000,0.000000,1.404300e+06,22.115000,17.000000,16723.125000,1.000000
50%,2016.0,47.601395,-122.318640,1.0,57708.000000,0.000000,1.982446e+06,52.415000,47.000000,20662.250000,1.000000
75%,2016.0,47.672288,-122.291575,1.0,92632.250000,0.000000,2.899628e+06,94.692500,62.500000,30711.791667,1.000000
max,2016.0,47.724630,-122.258640,1.0,289588.000000,17471.000000,1.356777e+07,359.090000,116.000000,80931.666667,3.000000


In [11]:
#  Récupération des dictionnaires complets renvoyés par l'API BentoML
predictions = [
    service.predict(BuildingInput(**row))
    for row in df_esti.to_dict(orient="records")
]

#  Assignation des prédictions dans le DataFrame de test
data_test = df_esti.copy()

# Extraction des deux prédictions
data_test["prediction_tCO2"] = [p["prediction_tCO2"] for p in predictions]
data_test["prediction_energy"] = [p["prediction_SiteEnergyUse_kBtu"] for p in predictions
]




total_co2_reel = data_test["TotalGHGEmissions"].sum()
total_co2_pred = data_test["prediction_tCO2"].sum()
ratio_co2_pct = (total_co2_pred / total_co2_reel) * 100

total_energy_reel = data_test["SiteEnergyUse(kBtu)"].sum()
total_energy_pred = data_test["prediction_energy"].sum()
ratio_energy_pct = (total_energy_pred / total_energy_reel) * 100


# ==============================================================================
# AFFICHAGE  DES RÉSULTATS
# ==============================================================================
print("=" * 60)
print("--- RÉSULTATS GLOBAUX DU MODÈLE (CO2 & ÉNERGIE) ---")
print("=" * 60)

print("\n1. ÉMISSIONS DE CO2 (TotalGHGEmissions)")
print(f"  - Volume réel total     : {total_co2_reel:,.2f} tCO2e")
print(f"  - Volume prédit total    : {total_co2_pred:,.2f} tCO2e")
print(f"  - Ratio global de volume : {ratio_co2_pct:.2f}%")


print("\n2. CONSOMMATION D'ÉNERGIE (SiteEnergyUse)")
print(f"  - Volume réel total     : {total_energy_reel:,.2f} kBtu")
print(f"  - Volume prédit total    : {total_energy_pred:,.2f} kBtu")
print(f"  - Ratio global de volume : {ratio_energy_pct:.2f}%")


--- RÉSULTATS GLOBAUX DU MODÈLE (CO2 & ÉNERGIE) ---

1. ÉMISSIONS DE CO2 (TotalGHGEmissions)
  - Volume réel total     : 7,832.04 tCO2e
  - Volume prédit total    : 9,767.50 tCO2e
  - Ratio global de volume : 124.71%

2. CONSOMMATION D'ÉNERGIE (SiteEnergyUse)
  - Volume réel total     : 259,971,904.69 kBtu
  - Volume prédit total    : 411,962,769.33 kBtu
  - Ratio global de volume : 158.46%
